# 05 SQL Business Analysis

## Objective

This notebook executes SQL business analysis queries using DuckDB on top of the Gold analytical layer.

The goal is to demonstrate how the project can use SQL to analyze customer behavior, customer segments, delivery experience, satisfaction, and category performance.

Main steps:

- Connect to DuckDB in memory.
- Read SQL files from the `sql/` folder.
- Execute each query against Parquet files from the Gold layer.
- Display business-ready analytical outputs.


## 1. Import libraries

In [ ]:
import duckdb
import pandas as pd

from pathlib import Path

## 2. Define paths

This notebook assumes it is located inside:

`01_customer_analytics/notebooks/`

Therefore, `..` points to the `01_customer_analytics/` project folder.

In [ ]:
PROJECT_PATH = Path("..")
SQL_PATH = PROJECT_PATH / "sql"
GOLD_PATH = PROJECT_PATH / "data" / "gold"

SQL_PATH, GOLD_PATH

## 3. Validate SQL files

In [ ]:
sql_files = sorted(SQL_PATH.glob("*.sql"))

for file in sql_files:
    print(file.name)

## 4. Create DuckDB connection

In [ ]:
con = duckdb.connect(database=":memory:")
con

## 5. Helper function to run SQL files

The SQL files use relative paths from the notebook working directory.

Example:

`read_parquet('../data/gold/customer_features.parquet')`

In [ ]:
def run_sql_file(file_name: str) -> pd.DataFrame:
    query_path = SQL_PATH / file_name
    query = query_path.read_text(encoding="utf-8")
    return con.execute(query).df()

## 6. Customer overview

In [ ]:
customer_overview = run_sql_file("01_customer_overview.sql")
customer_overview

### Interpretation notes

Use this output to summarize the overall customer base, including total customers, orders, revenue, average order value, repeat buyer share, and delayed order rate.

## 7. Customer segment analysis

In [ ]:
customer_segments_sql = run_sql_file("02_customer_segments.sql")
customer_segments_sql

### Interpretation notes

Use this output to identify which customer segments generate the most revenue, which segments have the highest average value, and how revenue is distributed across the customer base.

## 8. Delivery and satisfaction analysis

In [ ]:
delivery_satisfaction = run_sql_file("03_delivery_satisfaction.sql")
delivery_satisfaction

### Interpretation notes

Use this output to compare delayed versus on-time orders and quantify how delivery performance relates to review scores.

## 9. Category performance analysis

In [ ]:
category_performance = run_sql_file("04_category_performance.sql")
category_performance.head(20)

### Interpretation notes

Use this output to identify top product categories by revenue, order volume, review score, and delay rate.

## 10. Save SQL outputs for dashboarding

In [ ]:
OUTPUT_PATH = GOLD_PATH / "sql_outputs"
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

customer_overview.to_parquet(OUTPUT_PATH / "customer_overview.parquet", index=False)
customer_segments_sql.to_parquet(OUTPUT_PATH / "customer_segments_summary.parquet", index=False)
delivery_satisfaction.to_parquet(OUTPUT_PATH / "delivery_satisfaction.parquet", index=False)
category_performance.to_parquet(OUTPUT_PATH / "category_performance.parquet", index=False)

for file in sorted(OUTPUT_PATH.glob("*.parquet")):
    print(file.name)

## Conclusion

The SQL business analysis layer was created successfully.

The project now includes reusable SQL queries that analyze customer overview, customer segments, delivery satisfaction, and category performance on top of the Gold analytical layer.